# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

## Finding 1: The Anatomy of Growing Content

The paper reports that growing content is longer, younger, and has better visibility signals compared with declining content.

Methodology question:

How is the growth/decline label created?

The finding depends on how pages are classified as growing or declining.
A reviewer should verify whether the label is based on a consistent time window and whether external factors such as seasonality, search demand changes, or algorithm updates could influence the observed differences.

Does the validation design support the claim?

The large sample size supports that the relationship is measurable and directional. However, because the comparison is observational, it does not prove that increasing word count or reducing age directly causes growth.

The claim is appropriate as a directional insight rather than a causal statement.


## Finding 2: The Content Performance Curve

The paper reports that content performance changes across lifecycle stages and that older refreshed content can recover visibility.

Methodology question:

Where does the refresh label come from?

The analysis should clarify how refreshed pages are identified and whether refreshed pages are comparable with non-refreshed pages.

Does the validation design support the claim?

The comparison provides useful evidence about observed differences between content groups. However, refreshed pages may have received additional attention or resources, creating other explanations for performance changes.

The result supports a decision-support recommendation, but not a guaranteed impact from refreshing content.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


url = "https://raw.githubusercontent.com/estuchis21/ml-internship-week1/refs/heads/main/data/raw/content_refresh_anonymized.csv"


df = pd.read_csv(url)


df.head()
df["refresh_flag"] = (
    (df["trend_pct"] < -0.20) &
    (df["avg_position"] > 5)
).astype(int)


df["refresh_flag"].value_counts()

features = [

    "days_with_impressions",
    "word_count",
    "char_count",
    "days_since_last_update",
    "days_with_sessions",
    "ai_sessions_90d",
    "pageviews_90d",
    "users_90d",
    "sessions_90d",
    "clicks_90d",
    "impressions_90d",
    "clicks_last_30d",
    "impressions_last_30d",
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "engaged_sessions_90d",
    "scroll_events_90d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"

]


X = df[features]

y = df["refresh_flag"]



# 3. Before: Random split validation

The original model validation used a random train-test split.

This provides an initial estimate of model performance, but it may not fully represent future predictions because training and testing data can have similar distributions.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)


model = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),


    (
        "model",
        GradientBoostingClassifier(

            n_estimators=500,

            learning_rate=0.005,

            max_depth=6,

            random_state=42

        )
    )

])


model.fit(
    X_train,
    y_train
)


pred_random = model.predict(
    X_test
)


prob_random = model.predict_proba(
    X_test
)[:,1]

In [3]:
random_results = pd.DataFrame({

    "Metric":[

        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"

    ],

    "Score":[

        accuracy_score(
            y_test,
            pred_random
        ),

        precision_score(
            y_test,
            pred_random
        ),

        recall_score(
            y_test,
            pred_random
        ),

        f1_score(
            y_test,
            pred_random
        ),

        roc_auc_score(
            y_test,
            prob_random
        )

    ]

})


random_results

,Metric,Score
0,Accuracy,0.859333
1,Precision,0.819827
2,Recall,0.965839
3,F1,0.886863
4,ROC-AUC,0.922603


# 4. After: Time-aware validation

A time-aware split was added to simulate future predictions.

The model trains on one part of the content lifecycle and is evaluated on another part.

This provides a more realistic estimate because SEO data changes over time.

In [4]:
df_time = df.sort_values(
    "content_age_days"
)


split_point = int(
    len(df_time)*0.8
)


train = df_time.iloc[:split_point]

test = df_time.iloc[split_point:]


X_train_time = train[features]

y_train_time = train["refresh_flag"]


X_test_time = test[features]

y_test_time = test["refresh_flag"]

In [5]:
time_model = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),


    (
        "model",
        GradientBoostingClassifier(

            n_estimators=500,

            learning_rate=0.005,

            max_depth=6,

            random_state=42

        )
    )

])


time_model.fit(

    X_train_time,

    y_train_time

)


pred_time = time_model.predict(
    X_test_time
)


prob_time = time_model.predict_proba(
    X_test_time
)[:,1]

In [6]:
time_results = pd.DataFrame({

    "Metric":[

        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"

    ],

    "Score":[

        accuracy_score(
            y_test_time,
            pred_time
        ),

        precision_score(
            y_test_time,
            pred_time
        ),

        recall_score(
            y_test_time,
            pred_time
        ),

        f1_score(
            y_test_time,
            pred_time
        ),

        roc_auc_score(
            y_test_time,
            prob_time
        )

    ]

})


time_results

,Metric,Score
0,Accuracy,0.846833
1,Precision,0.811405
2,Recall,0.940000
3,F1,0.870981
4,ROC-AUC,0.923048


# 6. Leakage audit

The final feature set was reviewed for possible data leakage.

The target variable refresh_flag is created from trend_pct and avg_position.

The selected features represent historical information available before a refresh decision, including traffic, engagement, content age and freshness.

No feature was intentionally created from future outcomes.

However, because SEO data changes over time, time-aware validation is recommended.

# 7. Claim rewrite

## Original claim

"The model predicts which content should be refreshed."


## Safer claim

"The model observed historical patterns associated with content refresh signals. Its predictions provide decision-support by helping prioritize content items for human review. The model does not automatically decide which pages should be refreshed."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.